# VLM_DataGeneration — Sinh QA nháp bằng VLM (Bước ③)

Notebook này thực hiện **Bước ③ (Sinh QA nháp bằng VLM)** trong pipeline gán nhãn
VNTA-VQA: nhận clip đã tiền xử lý từ `../clip_pipeline` (bước ①–②), gửi cho
**Gemini** (`gemini-3-flash-preview` qua `google-genai`), và sinh ra bộ câu hỏi
trắc nghiệm nháp theo đúng 10 nhóm taxonomy (S, E, N, C, V, O, R, Attr, Prev, Con),
kèm `scene_description` và nhãn `answerable` sơ bộ cho từng câu.

Output của notebook (`vqa_dataset_gemini_v2.json`) là **QA nháp**, đi tiếp vào
Bước ④ (checklist soạn thảo) → Bước ⑤–⑧ (guideline, majority vote, GT) theo
`../docs/EG-TrafficQA-VN_report.md`. Đây KHÔNG phải ground truth — mọi câu hỏi
đều cần con người audit lại trước khi đưa vào benchmark.

Được viết để chạy trên **Google Colab** (dùng `google.colab.userdata` cho API key
và `google.colab.files` để upload video). Xem `README.md` trong thư mục này để
biết cách chạy trên Colab hoặc adapt sang máy local.

**Yêu cầu:** `GOOGLE_API_KEY` (Gemini API, có billing/quota cho video) đặt trong
Colab Secrets (icon chìa khoá bên trái).


In [ ]:
!pip install -q -U google-generativeai

In [ ]:
from google import genai
from google.colab import userdata

GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')
client = genai.Client(api_key=GOOGLE_API_KEY)

MODEL_NAME = "gemini-3-flash-preview"

In [ ]:
test_response = client.models.generate_content(
    model="gemini-3-flash-preview",
    contents="Xin chào, bạn có hoạt động không?"
)
print(test_response.text)

In [ ]:
!pip install -q -U google-genai

In [ ]:
from google.colab import files
import os

os.makedirs("videos", exist_ok=True)

print("⚠️ Trong hộp thoại chọn file: giữ Ctrl (Windows/Linux) hoặc Cmd (Mac) để chọn nhiều video cùng lúc")
uploaded = files.upload()

video_paths = []
for fname in uploaded.keys():
    dst = os.path.join("videos", fname)
    os.rename(fname, dst)
    video_paths.append(dst)

print(f"Đã upload {len(video_paths)} video:")
for p in video_paths:
    print(" -", p)

In [ ]:
PROMPT_TEMPLATE = """
Bạn là chuyên gia gán nhãn dữ liệu VQA cho tai nạn giao thông đường bộ tại Việt Nam.
Hãy xem THẬT KỸ video được cung cấp (chú ý từng khung hình, kể cả chi tiết nhỏ như
màu xe, biển báo, hành vi người đi đường) trước khi trả lời.

BƯỚC 1 — MÔ TẢ QUAN SÁT (bắt buộc, viết trước khi trả lời câu hỏi):
Mô tả ngắn gọn (4-6 câu) những gì bạn quan sát được: bối cảnh, loại đường, thời tiết/ánh sáng,
các phương tiện/người liên quan, toàn bộ diễn biến từ trước đến sau va chạm (nếu có),
hậu quả và phản ứng của các bên sau va chạm.

BƯỚC 2 — SINH CÂU HỎI:
Dựa trên phần mô tả ở Bước 1, sinh bộ câu hỏi trắc nghiệm theo đúng 10 nhóm dưới đây.

QUY TẮC BẮT BUỘC:
1. Mỗi nhóm sinh ĐÚNG 1 câu hỏi (tổng cộng 10 câu hỏi).
2. Mỗi câu hỏi có đúng 4 đáp án (A, B, C, D), chỉ 1 đáp án đúng.
3. QUAN TRỌNG: Chỉ chọn đáp án "Không thể xác định" khi THỰC SỰ không có bất kỳ manh mối
   nào trong video (ví dụ: bị che khuất hoàn toàn, quá tối, quá xa, camera cắt trước khi
   sự việc xảy ra). KHÔNG được chọn "Không thể xác định" chỉ vì không chắc chắn 100% —
   hãy dựa vào bằng chứng quan sát được (dù là gián tiếp) để suy luận ra đáp án hợp lý nhất,
   giống như một người xem video thực tế sẽ đoán.
4. 3 đáp án sai phải hợp lý, liên quan đến bối cảnh giao thông (không quá dễ loại trừ).
5. Với nhóm Attribution và Conclusion: chỉ nhận định dựa trên bằng chứng THỊ GIÁC quan sát
   được (vị trí, tốc độ, hướng di chuyển, tín hiệu...), KHÔNG đưa ra phán quyết pháp lý
   hay quy trách nhiệm chính thức — đây là suy luận mang tính phân tích tình huống, không
   phải kết luận lỗi của cơ quan chức năng.
6. Với nhóm Outcome: mô tả khách quan mức độ va chạm/hư hỏng quan sát được trên video,
   không suy đoán thương tích không nhìn thấy được.
7. Trả lời CHỈ bằng JSON hợp lệ theo schema bên dưới, không thêm text/markdown nào khác
   ngoài JSON (phần mô tả Bước 1 cũng đưa vào field "scene_description" trong JSON).

Các nhóm câu hỏi:
- S (Scene/Context): thời tiết, ánh sáng, loại đường, mặt đường.
- E (Entities): loại/màu phương tiện hoặc người liên quan.
- N (Narrative): tóm tắt toàn bộ sự kiện chính diễn ra trong video.
- C (Causal): nguyên nhân trực tiếp/hành động ngay trước va chạm.
- V (Violation): có vi phạm luật giao thông quan sát được không.
- O (Outcome): hậu quả quan sát được — mức độ va chạm, hư hỏng (mô tả khách quan).
- R (Response): hành vi sau tai nạn — dừng lại, bỏ chạy, hỗ trợ nạn nhân...
- Attr (Attribution): bên nào góp phần CHÍNH gây va chạm, dựa trên bằng chứng thị giác,
  không phán lỗi pháp lý.
- Prev (Prevention): lẽ ra người lái/người liên quan nên làm gì để tránh va chạm này.
- Con (Conclusion): phân tích bên nào có khả năng gây ra va chạm dựa trên tổng hợp bằng
  chứng thị giác quan sát được xuyên suốt video.

Schema JSON:
{
  "video_id": "<tên video>",
  "scene_description": "<mô tả Bước 1>",
  "questions": [
    {
      "category": "S|E|N|C|V|O|R|Attr|Prev|Con",
      "question": "...",
      "options": {"A": "...", "B": "...", "C": "...", "D": "..."},
      "correct_answer": "A|B|C|D",
      "reasoning_type": "perception|recognition|summarization|causal|rule_based|outcome_assessment|behavioral|attribution|preventive|conclusive",
      "answerable": "answerable|ambiguous|not_answerable"
    }
  ],
  "answerability_note": {
    "hardest_category": "S|E|N|C|V|O|R|Attr|Prev|Con",
    "explanation": "..."
  }
}

Tên video: {video_name}
"""

In [ ]:
import time
import json
import re

def upload_video_to_gemini(path):
    video_file = client.files.upload(file=path)
    while video_file.state.name == "PROCESSING":
        time.sleep(3)
        video_file = client.files.get(name=video_file.name)
    if video_file.state.name == "FAILED":
        raise RuntimeError(f"Upload thất bại: {path}")
    return video_file

def extract_json(text):
    match = re.search(r"\{.*\}", text, re.DOTALL)
    if not match:
        raise ValueError("Không tìm thấy JSON trong response")
    return json.loads(match.group(0))

def generate_vqa_for_video(path, fps=3):
    video_name = os.path.basename(path)
    print(f"→ Đang xử lý {video_name} (fps={fps}) ...")

    video_file = upload_video_to_gemini(path)
    prompt = PROMPT_TEMPLATE.replace("{video_name}", video_name)

    from google.genai import types

    video_part = types.Part(
        file_data=types.FileData(
            file_uri=video_file.uri,
            mime_type=video_file.mime_type,
        ),
        video_metadata=types.VideoMetadata(fps=fps),
    )

    response = client.models.generate_content(
        model=MODEL_NAME,
        contents=[video_part, prompt],
        config=types.GenerateContentConfig(
            temperature=0.3,
            response_mime_type="application/json",
        ),
    )

    try:
        data = extract_json(response.text)
    except Exception as e:
        print(f"  ⚠️ Lỗi parse JSON cho {video_name}: {e}")
        data = {"video_id": video_name, "raw_response": response.text}

    client.files.delete(name=video_file.name)
    return data

In [ ]:
results = []
for path in video_paths:
    try:
        result = generate_vqa_for_video(path, fps=3)
        results.append(result)
    except Exception as e:
        print(f"Lỗi với {path}: {e}")

with open("vqa_dataset_gemini_v2.json", "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

print(f"Đã sinh xong {len(results)} bộ VQA.")

In [ ]:
from IPython.display import display, HTML, Video

def show_result(video_path, result):
    display(HTML(f"<h3 style='color:#222;'>📹 {os.path.basename(video_path)}</h3>"))
    display(Video(video_path, embed=True, width=480))

    if "raw_response" in result:
        print("⚠️ Model không trả JSON hợp lệ:")
        print(result["raw_response"])
        return

    # Hiển thị mô tả cảnh để bạn đối chiếu model "nhìn thấy" gì
    desc = result.get("scene_description", "")
    display(HTML(f"""
    <div style='background:#fff8e1; padding:10px; border-radius:6px; color:#333; margin-bottom:8px;'>
        <b style='color:#000;'>👁️ Model quan sát:</b> {desc}
    </div>
    """))

    for q in result.get("questions", []):
        options_html = ""
        for key, val in q["options"].items():
            mark = "✅" if key == q["correct_answer"] else "❌"
            bold = "font-weight:bold;" if key == q["correct_answer"] else ""
            options_html += f"<div style='color:#111; {bold}'>({key}) {val} {mark}</div>"

        display(HTML(f"""
        <div style='border:1px solid #ccc; border-radius:8px; padding:12px; margin:8px 0;
                    background:#f9f9f9; color:#111;'>
            <b style='color:#000;'>[{q['category']}]</b>
            <span style='color:#111;'>{q['question']}</span><br>
            {options_html}
            <div style='color:#555; font-size:12px; margin-top:6px;'>
                reasoning: {q.get('reasoning_type','')} | answerable: {q.get('answerable','')}
            </div>
        </div>
        """))

    note = result.get("answerability_note", {})
    if note:
        display(HTML(f"""
        <div style='background:#eef6ff; padding:10px; border-radius:6px; color:#111;'>
            <b style='color:#000;'>🧩 Hardest category:</b> {note.get('hardest_category','')}<br>
            <i style='color:#333;'>{note.get('explanation','')}</i>
        </div>
        """))
    display(HTML("<hr>"))


for path, result in zip(video_paths, results):
    show_result(path, result)